# Módulo 4 - Gabarito - Lista de Exercícios

> Este notebook contém as soluções dos exercícios do Módulo 4 (EDA).
> Tente resolver sozinho antes de consultar este gabarito!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)

# Carregando os dados
colunas_adh = ["ano","id_municipio","idhm","idhm_e","idhm_r","idhm_l",
               "renda_pc","indice_gini","prop_pobreza","expectativa_anos_estudo",
               "taxa_analfabetismo_15_mais","expectativa_vida",
               "taxa_agua_encanada","taxa_energia_eletrica","populacao"]

for c in ["../eda/indices_adh_municipio.csv","eda/indices_adh_municipio.csv"]:
    if Path(c).exists():
        df_adh = pd.read_csv(c, usecols=lambda x: x in colunas_adh)
        break

for c in ["../eda/identificador_municipios.csv","eda/identificador_municipios.csv"]:
    if Path(c).exists():
        df_mun = pd.read_csv(c, usecols=["id_municipio","nome","sigla_uf","nome_regiao"])
        break

df = df_adh.merge(df_mun, on="id_municipio", how="left")
for col in [c for c in colunas_adh if c not in ["ano","id_municipio"]]:
    if col in df.columns:
        df[col] = df.groupby("ano")[col].transform(lambda x: x.fillna(x.median()))
df = df.dropna(subset=["idhm"])

print(f"Dados carregados: {df.shape}")
df_2010 = df[df["ano"]==2010].copy()

## Exercício 1 - Evolução estadual de 5 estados

In [ ]:
# Seleciona 5 estados (um de cada região)
estados = ["SP", "CE", "AM", "RS", "GO"]

df_estados = df[df["sigla_uf"].isin(estados)].copy()
ev_estados = df_estados.groupby(["ano","sigla_uf"])["idhm"].mean().reset_index()

plt.figure(figsize=(12, 6))
for estado in estados:
    d = ev_estados[ev_estados["sigla_uf"]==estado]
    plt.plot(d["ano"], d["idhm"], marker="o", linewidth=2, label=estado, markersize=8)

plt.title("Evolução do IDHM médio em 5 estados (1991-2010)", fontsize=14)
plt.xlabel("Ano")
plt.ylabel("IDHM médio")
plt.xticks([1991, 2000, 2010])
plt.legend(title="Estado")
plt.tight_layout()
plt.show()

# Qual estado mais avançou?
for estado in estados:
    d = ev_estados[ev_estados["sigla_uf"]==estado].sort_values("ano")
    crescimento = d["idhm"].iloc[-1] - d["idhm"].iloc[0]
    print(f"{estado}: crescimento = {crescimento:.3f}")

## Exercício 2 - Correlações em 1991

In [ ]:
df_1991 = df[df["ano"]==1991].copy()

colunas_corr = [c for c in [
    "idhm","renda_pc","indice_gini","expectativa_vida",
    "taxa_analfabetismo_15_mais","prop_pobreza",
    "taxa_agua_encanada","taxa_energia_eletrica"
] if c in df_1991.columns]

corr = df_1991[colunas_corr].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Mapa de correlação - dados de 1991", fontsize=14)
plt.tight_layout()
plt.show()

print("\n As correlações com o IDHM em 1991:")
print(corr["idhm"].drop("idhm").sort_values(key=abs, ascending=False))

## Exercício 3 - Top e Bottom 10 municípios

In [ ]:
top10 = df_2010.nlargest(10, "idhm")[['nome','sigla_uf','nome_regiao','idhm']]
bot10 = df_2010.nsmallest(10, "idhm")[['nome','sigla_uf','nome_regiao','idhm']]

print("TOP 10 - Maior IDHM (2010):")
print(top10.to_string(index=False))

print("\nBOTTOM 10 - Menor IDHM (2010):")
print(bot10.to_string(index=False))

print("\nConcentração por região - Top 10:")
print(top10["nome_regiao"].value_counts())
print("\nConcentração por região - Bottom 10:")
print(bot10["nome_regiao"].value_counts())

## Exercício 4 - Pobreza e IDHM

In [ ]:
if "prop_pobreza" in df_2010.columns:
    plt.figure(figsize=(12,6))
    sns.scatterplot(data=df_2010, x="prop_pobreza", y="idhm",
                   hue="nome_regiao", alpha=0.4, palette="Set2", s=20)
    plt.title("Proporção de pobreza x IDHM por município (2010)", fontsize=14)
    plt.xlabel("Proporção da população em situação de pobreza (%)")
    plt.ylabel("IDHM")
    plt.legend(title="Região", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

    corr = df_2010[["prop_pobreza","idhm"]].corr().iloc[0,1]
    print(f"\n Correlação: {corr:.3f}")
    print("Correlação negativa forte: mais pobreza = menor desenvolvimento humano.")

## Exercício 5 (Desafio) - Ranking estadual por IDHM

In [ ]:
ranking = df_2010.groupby(["sigla_uf","nome_regiao"])["idhm"].mean().reset_index()
ranking = ranking.sort_values("idhm", ascending=False).reset_index(drop=True)
ranking.index += 1
ranking.columns = ["Estado","Região","IDHM_médio"]

cores_regiao = {
    "Sul": "steelblue", "Sudeste": "royalblue",
    "Centro-Oeste": "seagreen", "Norte": "tomato", "Nordeste": "coral"
}

cores = [cores_regiao.get(r, "gray") for r in ranking["Região"]]

plt.figure(figsize=(12, 10))
plt.barh(ranking["Estado"], ranking["IDHM_médio"], color=cores, edgecolor="white")
plt.title("Ranking estadual por IDHM médio (2010)", fontsize=14)
plt.xlabel("IDHM médio")
plt.gca().invert_yaxis()

# Legenda manual
from matplotlib.patches import Patch
legenda = [Patch(facecolor=v, label=k) for k,v in cores_regiao.items()]
plt.legend(handles=legenda, title="Região", loc="lower right")
plt.tight_layout()
plt.show()

print("\n Top 5 estados:")
print(ranking.head(5)[["Estado","Região","IDHM_médio"]].to_string(index=True))
print("\n Bottom 5 estados:")
print(ranking.tail(5)[["Estado","Região","IDHM_médio"]].to_string())